# LinUCB-PSI on MNIST


In [7]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from tqdm import trange

from envs.mnist_env import load_mnist, group_by_class, MNISTBanditEnv
from models import LinUCBwithPSI_rank1


In [6]:
images, labels = load_mnist("../datasets/MNIST/raw")
features = images / np.max(np.linalg.norm(images, axis=1))
clusters = group_by_class(features, labels)

print(features.shape)
print([clusters[k].shape for k in range(10)])


FileNotFoundError: [Errno 2] No such file or directory: '../datasets/MNIST/raw'

In [ ]:
def psi(clusters, T=4000, lambd=0.02, beta=0.01, m=10, seed=0):
    np.random.seed(seed)

    env = MNISTBanditEnv(clusters, target_class=9)
    bandit = LinUCBwithPSI_rank1(num_arms=10, d=784, epsilon=lambd, alpha=beta, rank=m)
    for t in trange(T):
        contexts = env.get_contexts()
        scores = [bandit.score(contexts[a], a) for a in range(10)]
        action = int(np.argmax(scores))
        reward = env.step(action)
        bandit.update(contexts[action], action, reward)

    return env.cumulative_mistakes


In [ ]:
T = 4000
n_runs = 5
m = 64
lambd = 1
beta = 1

all_mistakes = []

for run in range(n_runs):
    print(f"\nRun {run+1}/{n_runs}")
    mistakes = psi(clusters, T=T, lambd=lambd, beta=beta, m=m, seed=run)
    all_mistakes.append(mistakes)

all_mistakes = np.array(all_mistakes)
mean_mistakes = np.mean(all_mistakes, axis=0)
std_mistakes = np.std(all_mistakes, axis=0)

print(f"num mistakes {mean_mistakes[-1]:.1f}")


In [ ]:
plt.figure(figsize=(10, 6))

rounds = np.arange(1, T + 1)

plt.plot(rounds, mean_mistakes, label='PSI', color='red', linewidth=2)
plt.fill_between(rounds,
                 mean_mistakes - std_mistakes,
                 mean_mistakes + std_mistakes,
                 alpha=0.2, color='red')

plt.xlabel('Number of Rounds', fontsize=12)
plt.ylabel('Mistakes', fontsize=12)
plt.title(f'PSI, rank = {m}', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
